# 00 — Fetch Submissions
Reads submission URLs from `scraped_data/submission_urls.txt`, fetches full post data
from the [Arctic Shift API](https://arctic-shift.photon-reddit.com) in batches of 500, and saves the result.

In [5]:
import re
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

/opt/homebrew/anaconda3/envs/RedditAnalysis/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [2]:
URLS_FILE   = Path("scraped_data/submission_urls.txt")
OUTPUT_FILE = Path("pickles/fetched_submissions.pkl")

API_BASE    = "https://arctic-shift.photon-reddit.com"
BATCH_SIZE  = 500   # API hard limit for /api/posts/ids
RETRY_WAIT  = 5     # seconds to wait before retrying a failed batch
MAX_RETRIES = 3

## Load URLs & extract post IDs

In [3]:
_ID_RE = re.compile(r"/comments/([A-Za-z0-9]+)")

def extract_post_id(url: str) -> str | None:
    """Extract Reddit base-36 post ID from a submission URL."""
    m = _ID_RE.search(url)
    return m.group(1) if m else None

raw_urls = URLS_FILE.read_text().splitlines()
raw_urls = [u.strip() for u in raw_urls if u.strip()]

url_to_id = {url: extract_post_id(url) for url in raw_urls}
failed_parse = [u for u, pid in url_to_id.items() if pid is None]

if failed_parse:
    print(f"WARNING: Could not parse {len(failed_parse)} URL(s):")
    for u in failed_parse[:5]:
        print(" ", u)

post_ids = list({pid for pid in url_to_id.values() if pid})
print(f"Loaded {len(raw_urls)} URLs → {len(post_ids)} unique post IDs")

  self
Loaded 87155 URLs → 87154 unique post IDs


## Fetch from Arctic Shift API

In [6]:
def fetch_batch(ids: list[str], session: requests.Session) -> list[dict]:
    """Fetch one batch of posts by ID; returns list of post dicts."""
    params = {"ids": ",".join(ids)}
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(f"{API_BASE}/api/posts/ids", params=params, timeout=30)
            if resp.status_code == 429:
                retry_after = int(resp.headers.get("Retry-After", RETRY_WAIT))
                print(f"Rate-limited — waiting {retry_after}s")
                time.sleep(retry_after)
                continue
            resp.raise_for_status()
            data = resp.json()
            return data.get("data", [])
        except requests.RequestException as exc:
            print(f"Attempt {attempt}/{MAX_RETRIES} failed: {exc}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_WAIT)
    return []  # return empty list after all retries exhausted


batches = [post_ids[i : i + BATCH_SIZE] for i in range(0, len(post_ids), BATCH_SIZE)]
print(f"Processing {len(batches)} batch(es) of up to {BATCH_SIZE} posts each")

all_posts: list[dict] = []
with requests.Session() as session:
    for batch in tqdm(batches, desc="Fetching batches"):
        posts = fetch_batch(batch, session)
        all_posts.extend(posts)

print(f"\nFetched {len(all_posts)} posts total")
missing = len(post_ids) - len(all_posts)
if missing > 0:
    print(f"  {missing} post(s) not returned by API (deleted/removed)")

Processing 175 batch(es) of up to 500 posts each



Fetching batches: 100%|██████████| 175/175 [03:03<00:00,  1.05s/it]


Fetched 87154 posts total


## Build DataFrame & save

In [7]:
df = pd.DataFrame(all_posts)
print(f"Shape: {df.shape}")
print(df.dtypes)
df.head(2)

Shape: (87154, 125)
all_awardings           object
allow_live_comments     object
archived                object
author                  object
author_created_utc     float64
                        ...   
rte_mode                object
call_to_action          object
previous_visits         object
poll_data               object
media_metadata          object
Length: 125, dtype: object


,all_awardings,allow_live_comments,archived,author,author_created_utc,author_flair_background_color,author_flair_css_class,author_flair_richtext,author_flair_template_id,author_flair_text,...,post_hint,preview,previous_selftext,author_cakeday,post_categories,rte_mode,call_to_action,previous_visits,poll_data,media_metadata
0,[],False,False,RxtAndrx06,1.563644e+09,None,None,[],None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,False,Kingkept,1.301427e+09,None,None,[],None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_pickle(OUTPUT_FILE)
print(f"Saved → {OUTPUT_FILE}")

Saved → pickles/fetched_submissions.pkl
